# RetailMind AI - Day 6: CNN Image Classification

Prototype a convolutional neural network on Fashion-MNIST, evaluate category confusions, and save the first computer-vision catalog classifier. This notebook uses TensorFlow/Keras for the Day 6 increment.

## 1. Import TensorFlow / Keras

In [1]:
import tensorflow as tf
from tensorflow import keras

tf.keras.utils.set_random_seed(42)
print('TensorFlow version:', tf.__version__)


I0000 00:00:1794157492.113562 1309958 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794157492.156531 1309958 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794157493.725662 1309958 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow version: 2.21.0


## 2. Load Fashion-MNIST

Fashion-MNIST is used as the fast prototype dataset before moving to real Fashion Product Images.

In [2]:
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

print('Training images:', x_train.shape)
print('Test images:', x_test.shape)
print('Classes:', len(class_names))


29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training images: (60000, 28, 28)
Test images: (10000, 28, 28)
Classes: 10


## 3. Normalise and reshape the images

In [3]:
x_train = (x_train.astype('float32') / 255.0)[..., tf.newaxis]
x_test = (x_test.astype('float32') / 255.0)[..., tf.newaxis]

print('CNN input shape:', x_train.shape)


CNN input shape: (60000, 28, 28, 1)


## 4. Add a small augmentation pipeline

The augmentation layer prepares the vision workflow for real catalog images while keeping the Fashion-MNIST prototype lightweight.

In [4]:
augmentation = keras.Sequential([
    keras.layers.RandomTranslation(0.05, 0.05),
    keras.layers.RandomRotation(0.05),
], name='augmentation')


E0000 00:00:1794157500.240384 1309958 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## 5. Build the CNN: conv → pool → dropout → dense

In [5]:
cnn = keras.Sequential([
    keras.layers.Input(shape=(28, 28, 1)),
    augmentation,
    keras.layers.Conv2D(32, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Dropout(0.25),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.30),
    keras.layers.Dense(10, activation='softmax')
])

cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ augmentation (Sequential)       │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       102,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 121,930 (476.29 KB)

 Trainable params: 121,930 (476.29 KB)

 Non-trainable params: 0 (0.00 B)

## 6. Train with validation monitoring

In [6]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = cnn.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=15,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

print('Epochs run:', len(history.history['loss']))
print('Best validation accuracy:', round(max(history.history['val_accuracy']), 4))


Epoch 1/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 39s 50ms/step - accuracy: 0.7050 - loss: 0.7905 - val_accuracy: 0.8062 - val_loss: 0.5010
Epoch 2/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 37s 50ms/step - accuracy: 0.7840 - loss: 0.5713 - val_accuracy: 0.8300 - val_loss: 0.4546
Epoch 3/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 37s 50ms/step - accuracy: 0.8051 - loss: 0.5191 - val_accuracy: 0.8548 - val_loss: 0.3988
Epoch 4/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 38s 51ms/step - accuracy: 0.8186 - loss: 0.4836 - val_accuracy: 0.8622 - val_loss: 0.3745
Epoch 5/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 37s 50ms/step - accuracy: 0.8298 - loss: 0.4563 - val_accuracy: 0.8565 - val_loss: 0.3766
Epoch 6/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 37s 49ms/step - accuracy: 0.8354 - loss: 0.4410 - val_accuracy: 0.8665 - val_loss: 0.3605
Epoch 7/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 37s 49ms/step - accuracy: 0.8425 - loss: 0.4215 - val_accuracy: 0.8765 - val_loss: 0.3362
Epoch 8/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 37s 50ms/step - accuracy: 0.8486 - loss: 0.4061 - 

## 7. Evaluate catalog-classification accuracy

In [7]:
test_loss, test_accuracy = cnn.evaluate(x_test, y_test, verbose=0)
print('Test loss:', round(test_loss, 4))
print('Test accuracy:', round(test_accuracy, 4))


Test loss: 0.3143
Test accuracy: 0.8854


## 8. Confusion matrix

Rows are actual categories and columns are predicted categories. The largest off-diagonal cells show which product categories the CNN confuses most often.

In [8]:
probabilities = cnn.predict(x_test, verbose=0)
predictions = tf.argmax(probabilities, axis=1, output_type=tf.int64)
cm = tf.math.confusion_matrix(y_test, predictions, num_classes=10)

print('Class order:')
for i, name in enumerate(class_names):
    print(i, name)

print('\nConfusion matrix:')
print(cm.numpy())


Class order:
0 T-shirt/top
1 Trouser
2 Pullover
3 Dress
4 Coat
5 Sandal
6 Shirt
7 Sneaker
8 Bag
9 Ankle boot

Confusion matrix:
[[801   0  18  32   5   1 136   0   7   0]
 [  0 972   1  16   4   0   5   0   2   0]
 [ 11   0 833   7  45   0 101   0   3   0]
 [ 11   9   7 904  15   1  49   0   4   0]
 [  1   0  82  39 773   0 104   0   1   0]
 [  0   0   0   0   0 990   0  10   0   0]
 [101   0  74  25  75   0 712   0  13   0]
 [  0   0   0   0   0  23   0 971   0   6]
 [  1   0   9   3   3   3   7   5 969   0]
 [  0   0   0   0   0   9   0  62   0 929]]


## 9. Find the most-confused category pairs

In [9]:
cm_no_diag = tf.linalg.set_diag(cm, tf.zeros(10, dtype=cm.dtype))
flat = tf.reshape(cm_no_diag, [-1])
top = tf.math.top_k(flat, k=5)

print('Top category confusions:')
for value, index in zip(top.values.numpy(), top.indices.numpy()):
    actual = index // 10
    predicted = index % 10
    print(f'{class_names[actual]} -> {class_names[predicted]}: {value}')


Top category confusions:
T-shirt/top -> Shirt: 136
Coat -> Shirt: 104
Pullover -> Shirt: 101
Shirt -> T-shirt/top: 101
Coat -> Pullover: 82


## 10. Save the first CV module

In [10]:
cnn.save('../models/catalog_cnn.keras')
print('Saved: ../models/catalog_cnn.keras')


Saved: ../models/catalog_cnn.keras


## 11. README-ready Day 6 summary

In [11]:
print('DAY 6 - CNN IMAGE CLASSIFICATION')
print('-' * 60)
print('Dataset: Fashion-MNIST')
print('Epochs run:', len(history.history['loss']))
print('Best validation accuracy:', f"{max(history.history['val_accuracy']):.4f}")
print('Test accuracy:', f"{test_accuracy:.4f}")
print('Test loss:', f"{test_loss:.4f}")
print('Artifact: models/catalog_cnn.keras')
print('\nTop category confusions:')
for value, index in zip(top.values.numpy(), top.indices.numpy()):
    actual = index // 10
    predicted = index % 10
    print(f'{class_names[actual]} -> {class_names[predicted]}: {value}')


DAY 6 - CNN IMAGE CLASSIFICATION
------------------------------------------------------------
Dataset: Fashion-MNIST
Epochs run: 15
Best validation accuracy: 0.8903
Test accuracy: 0.8854
Test loss: 0.3143
Artifact: models/catalog_cnn.keras

Top category confusions:
T-shirt/top -> Shirt: 136
Coat -> Shirt: 104
Pullover -> Shirt: 101
Shirt -> T-shirt/top: 101
Coat -> Pullover: 82
